# Frequency-domain analysis

The periodogram redistributes a series' variance across cycle lengths. This notebook builds a periodogram by hand with the FFT, converts peaks to periods, shows spectral leakage and tapering, aliasing, and the theoretical AR(1) spectrum.

Companion note: [frequency_domain_analysis.md](../../notes/time_series/frequency_domain_analysis.md).

In [1]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
plt.rcParams['figure.figsize'] = (8, 3.5)

N = 240
t = np.arange(N)
x = 1.8 * np.sin(2 * np.pi * t / 12) + 0.8 * np.sin(2 * np.pi * t / 30) + rng.normal(size=N)
X = np.fft.rfft(x - x.mean())
freqs = np.fft.rfftfreq(N, d=1.0)
I = np.abs(X) ** 2 / N
top = np.argsort(I[1:])[::-1][:3] + 1
for k in top:
    print(f'bin k={k}  f={freqs[k]:.4f}  period={1/freqs[k]:.1f}  I={I[k]:.1f}')
plt.stem(freqs, I); plt.xlabel('cycles per observation'); plt.ylabel('periodogram'); plt.show()

bin k=20  f=0.0833  period=12.0  I=195.0
bin k=8  f=0.0333  period=30.0  I=43.4
bin k=105  f=0.4375  period=2.3  I=4.7


/tmp/ipykernel_78756/2112574260.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.stem(freqs, I); plt.xlabel('cycles per observation'); plt.ylabel('periodogram'); plt.show()


Peaks sit at $f=1/12\approx0.0833$ (bin $k=N f=20$) and $f=1/30=0.0333$ (bin 8). A peak is read as a period, $1/f$ observations per cycle, not as "frequency 12".

In [2]:
from scipy.signal import get_window
M = 250                     # not an integer number of 12-cycles -> leakage
tt = np.arange(M)
s = np.sin(2 * np.pi * tt / 12.7)
def spec(sig, win):
    w = get_window(win, len(sig))
    P = np.abs(np.fft.rfft(sig * w, n=4096)) ** 2
    return np.fft.rfftfreq(4096), P / P.max()
for win in ['boxcar', 'hann']:
    f, P = spec(s, win)
    plt.semilogy(f, P + 1e-12, label=win)
plt.ylim(1e-8, 2); plt.xlim(0, 0.3); plt.legend(); plt.title('leakage: rectangular vs Hann window'); plt.show()

/tmp/ipykernel_78756/284515766.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.ylim(1e-8, 2); plt.xlim(0, 0.3); plt.legend(); plt.title('leakage: rectangular vs Hann window'); plt.show()


The rectangular window has the narrowest main lobe but high side lobes; the Hann taper suppresses side lobes at the cost of a wider peak. Next, aliasing: a cosine at $0.65$ cycles per sample is indistinguishable at integer times from one at $0.35$.

In [3]:
tt = np.arange(40)
hi = np.cos(2 * np.pi * 0.65 * tt)
lo = np.cos(2 * np.pi * 0.35 * tt)
print('identical samples:', np.allclose(hi, lo))
tc = np.linspace(0, 10, 1000)
plt.plot(tc, np.cos(2 * np.pi * 0.65 * tc), alpha=.5, label='0.65 cycles')
plt.plot(tc, np.cos(2 * np.pi * 0.35 * tc), alpha=.5, label='0.35 cycles')
plt.plot(tt[:11], hi[:11], 'ko', label='samples'); plt.legend(loc='lower left'); plt.show()

identical samples: True


/tmp/ipykernel_78756/3791565615.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.plot(tt[:11], hi[:11], 'ko', label='samples'); plt.legend(loc='lower left'); plt.show()


Finally, the spectrum of an AR(1) is $f(\omega)=\sigma^2/[2\pi(1-2\phi\cos\omega+\phi^2)]$; compare it with a smoothed periodogram, and check the first-difference gain $4\sin^2(\omega/2)$.

In [4]:
from scipy.signal import welch
phi = 0.7
y = np.zeros(4000 + 100)
e = rng.normal(size=len(y))
for i in range(1, len(y)):
    y[i] = phi * y[i-1] + e[i]
y = y[100:]
f, P = welch(y, nperseg=256, detrend='constant')
w = 2 * np.pi * f
theory = 1 / (1 - 2 * phi * np.cos(w) + phi ** 2)     # sigma^2=1, one-sided scale 2/(2*pi)*2*pi
plt.semilogy(f, P, label='Welch estimate')
plt.semilogy(f, 2 * theory, '--', label='theory (one-sided, scaled)')
plt.legend(); plt.xlabel('cycles per observation'); plt.show()
om = np.linspace(0, np.pi, 5)
print('difference gain |1-e^{-iw}|^2:', np.round(np.abs(1 - np.exp(-1j * om)) ** 2, 4))
print('4 sin^2(w/2)                 :', np.round(4 * np.sin(om / 2) ** 2, 4))

difference gain |1-e^{-iw}|^2: [0.     0.5858 2.     3.4142 4.    ]
4 sin^2(w/2)                 : [0.     0.5858 2.     3.4142 4.    ]


/tmp/ipykernel_78756/2612200751.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.legend(); plt.xlabel('cycles per observation'); plt.show()
